# The test dataset:

https://demo.borealisdata.ca/dataset.xhtml?persistentId=doi%3A10.80240%2FFK2%2FUWHWJT&version=DRAFT

# Change to the Juypter Server directory

In [8]:
import os
import sys
import yaml
import pandas as pd
import datetime
import janitor # https://pyjanitor.readthedocs.io/
import hashlib
import subprocess
import pydatacuration.utils as utils
import time
import shutil
import datetime
import dotenv
from pyDataverse.api import NativeApi

In [2]:
# Load the configuration file
with open('config.yaml', 'r') as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

In [3]:
# Create log files directory
utils.mk_log_dir()

# Output

## Tree-structure of the dataset

In [4]:
# Redirect Tree structure to a text file
original_stdout = sys.stdout # Save a reference to the original standard output
with open('./log_files/ds_structure.txt', 'w') as f:
    sys.stdout = f # Change the standard output to the file we created.
    utils.list_files('raw_data')
    sys.stdout = original_stdout # Reset the standard output to its original value

## Basic log file

In [5]:
def get_sha256sum(file_path):
    sha256 = hashlib.sha256()
    with open(file_path, 'rb') as f:
        for block in iter(lambda: f.read(4096), b''):
            sha256.update(block)
    return sha256.hexdigest()


def get_hidden_file(file_path):
    return os.path.basename(file_path).startswith('.')

In [6]:
def get_filepaths_and_metadata(directory):
    file_info = []
    id = 1
    for root, dirs, files in os.walk(directory):
        for file in files:
            full_file_path = os.path.join(root, file)
            parent_directory = os.path.basename(os.path.dirname(full_file_path))
            if os.path.exists(full_file_path):
                created = datetime.datetime.fromtimestamp(os.path.getctime(full_file_path))
                modified = datetime.datetime.fromtimestamp(os.path.getmtime(full_file_path))
                file_extension = os.path.splitext(full_file_path)[1]
                sha256_hash = get_sha256sum(full_file_path)
            else:
                created = None
                modified = None
                file_extension = None
                sha256_hash = None
            
            file_info.append({
                'ds.file_id': id,
                'ds.root': directory,
                'ds.parent_directory': parent_directory,
                'ds.depth': full_file_path.count(os.sep) - directory.count(os.sep) + 1,
                'ds.file_name': file,
                'ds.file_path': full_file_path,
                'ds.created': created,
                'ds.modified': modified,
                'ds.file_extension': file_extension,
                'ds.sha256_hash': sha256_hash,
            })
            id += 1
    return file_info

In [ ]:
# Directory to scan
directory = 'raw_data/'

# Get file paths and metadata
file_metadata = get_filepaths_and_metadata(directory)

# Create DataFrame
df = pd.DataFrame(file_metadata)

# Define the order of columns
df = df.reorder_columns(['ds.file_id', 'ds.root'] + [col for col in df.columns if col not in ['ds.file_id', 'ds.root']])

# Convert file_id to integer
df['ds.file_id'] = df['ds.file_id'].astype(int)

In [ ]:
# Export to CSV
df.to_csv('./log_files/ds_file_info.csv', index=False)

## OPF-Fido log

In [ ]:
# Combined command to change directory and run the fido command
command = 'cd ~/MDL/pydatacuration/ && fido -recurse -zip raw_data/ > log_files/temp_data/fileFormats_temp.csv'

# Run the command
subprocess.run(command, shell=True)

# Sleep for 2 seconds to allow the command to finish
time.sleep(2)

In [ ]:
file_formats_df = pd.read_csv('log_files/temp_data/fileFormats_temp.csv')
os.remove('log_files/temp_data/fileFormats_temp.csv')
shutil.rmtree('log_files/temp_data', ignore_errors=True)
file_formats_df.columns = ['fido.status', 'fido.info.time', 'fido.info.puid', 'fido.info.formatname', 'fido.info.signaturename', 'fido.info.filesize', 'fido.info.filename', 'fido.info.mimetype', 'fido.info.matchtype'] 

In [ ]:
# Prepare the DataFrames (if necessary)
df['ds.file_path'] = df['ds.file_path'].str.strip()
file_formats_df['fido.info.filename'] = file_formats_df['fido.info.filename'].str.strip()

# Perform a left join
dataCuration_df = pd.merge(df, file_formats_df, left_on='ds.file_path', right_on='fido.info.filename', how='left')

In [ ]:
# Check wehther the directory contains blank spaces
dataCuration_df['cur.dir.blank'] = dataCuration_df['ds.file_path'].apply(lambda x: 'TRUE' if ' ' in os.path.dirname(x) else 'FALSE')

# Check whether the file contains blank spaces
dataCuration_df['cur.file_name.blank'] = dataCuration_df['ds.file_name'].apply(lambda x: 'TRUE' if ' ' in x else 'FALSE')

# Check whether the file is a hidden file
dataCuration_df['cur.file.hidden'] = dataCuration_df['ds.file_path'].apply(lambda x: 'TRUE' if get_hidden_file(x) else 'FALSE')

In [ ]:
# Remove the duplicate column
dataCuration_df.drop(columns=['fido.info.filename'], inplace=True)

# Export to CSV
dataCuration_df.to_csv('./log_files/dataCuration.csv', index=False)

In [ ]:
with open('./log_files/ds_structure.txt', 'r') as f:
    ds_structure_file_content = f.read()

In [ ]:
markdown_text = f"""
# Project information

- Ticket Number: {config['project']['ticket-number']}

- Data curator: {config['curator']['name']}

- Data curator email: {config['curator']['email']}

- Date: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

# Submission details

- Data submitter: {config['submission']['submitter']['name']}

- Contact: {config['submission']['submitter']['email']}

- Date: {config['submission']['date']}


# Dataset details:

- Number of files: {df['ds.file_id'].count()}

- Total size: {df['ds.file_path'].apply(lambda x: os.path.getsize(x)).sum()/(1024*1024):.2f} MB


# File list
{dataCuration_df.to_markdown()}

# Dataset Structure
```
{ds_structure_file_content}
```

# Associated files
1. [Basic File Information](./log_files/ds_file_info.csv)
2. [Detailed File Information](./log_files/dataCuration.csv)
3. [Dataset Structure](./log_files/ds_structure.txt)

"""

In [ ]:
with open('./log_files/output.md', 'w') as f:
    f.write(markdown_text)

# Query